In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

/var/folders/mp/s7vyjtmd5dj1l_2fflblh8zh0000gn/T/ipykernel_39020/4057491872.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [9]:
def extract_text(response) -> str:
    """ Normalizes an LLM response's content into a plain string.

    Some models (e.g. Gemini) can return `content` as a list of content
    blocks (dicts with a 'text' key, or plain strings) instead of a str.
    """
    if not response:
        return "No summary available."

    content = response.content
    if isinstance(content, list):
        text = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    else:
        text = content

    return text.strip() if text else "No summary available."

In [3]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db", sample_rows_in_table_info=0)

def get_schema(_):
    return db.get_table_info()


def run_query(query):
    print(f'Query being run: {query} \n\n')
    return db.run(query)

In [4]:
print(get_schema(''))


CREATE TABLE albums (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES artists ("ArtistId")
)


CREATE TABLE artists (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)


CREATE TABLE customers (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES employees ("EmployeeId")
)


CREATE TABLE employees (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETI

In [5]:
def get_llm(load_from_hugging_face=False):
    if load_from_hugging_face:
        llm = HuggingFaceEndpoint(
            repo_id="Qwen/Qwen2.5-VL-7B-Instruct",
            task="text-generation",
            provider="hyperbolic",  # set your provider here
        )

        return ChatHuggingFace(llm=llm)
    
    return ChatGoogleGenerativeAI(model="gemini-3-flash-preview")


def write_sql_query(llm):
    template = """Based on the table schema below, write a SQL query that would answer the user's question:
    {schema}

    Question: {question}
    SQL Query:"""

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "Given an input question, convert it to a SQL query. No pre-amble. "
            "Please do not return anything else apart from the SQL query, no prefix aur suffix quotes, no sql keyword, nothing please"),
            ("human", template),
        ]
    )

    return (
        RunnablePassthrough.assign(schema=get_schema)
        | prompt
        | llm
        | StrOutputParser()
    )

In [6]:
def answer_user_query(query, llm):
    template = """Based on the table schema below, question, sql query, and sql response, write a natural language response:
    {schema}

    Question: {question}
    SQL Query: {query}
    SQL Response: {response}"""

    prompt_response = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Given an input question and SQL response, convert it to a natural language answer. No pre-amble.",
            ),
            ("human", template),
        ]
    )

    full_chain = (
        RunnablePassthrough.assign(query=write_sql_query(llm))
        | RunnablePassthrough.assign(
            schema=get_schema,
            response=lambda x: run_query(x["query"]),
        )
        | prompt_response
        | llm
    )

    return full_chain.invoke({"question": query})

In [ ]:
load_dotenv()

# write_sql_query(llm=get_llm(load_from_hugging_face=True)).invoke({"question": "Give me 10 Artists"})
query = 'Give some Tracks by the Artist name Audioslave'
response = answer_user_query(query, llm=get_llm(load_from_hugging_face=True))
print(response.content)

Query being run: SELECT T.Name FROM tracks T JOIN albums A ON T.AlbumId = A.AlbumId JOIN artists AR ON A.ArtistId = AR.ArtistId WHERE AR.Name = 'Audioslave' 


[{'type': 'text', 'text': 'Some tracks by the artist Audioslave include "Cochise", "Show Me How to Live", "Gasoline", "What You Are", "Like a Stone", "Set It Off", "Shadow on the Sun", "I am the Highway", "Exploder", "Hypnotize", "Bring\'em Back Alive", "Light My Way", "Getaway Car", "The Last Remaining Light", "Your Time Has Come", "Out Of Exile", "Be Yourself", "Doesn\'t Remind Me", "Drown Me Slowly", "Heaven\'s Dead", "The Worm", "Man Or Animal", "Yesterday To Tomorrow", "Dandelion", "#1 Zero", "The Curse", "Revelations", "One and the Same", "Sound of a Gun", "Until We Fall", "Original Fire", "Broken City", "Somedays", "Shape of Things to Come", "Jewel of the Summertime", "Wide Awake", "Nothing Left to Say But Goodbye", and "Moth".', 'extras': {'signature': 'EpwbCpkbARFNMg9yYV8P8eKnz42pSiBUXyWlYumhFfr78XE9e6De/c/RAhKHU76M26bp

In [10]:
extract_text(response)

'Some tracks by the artist Audioslave include "Cochise", "Show Me How to Live", "Gasoline", "What You Are", "Like a Stone", "Set It Off", "Shadow on the Sun", "I am the Highway", "Exploder", "Hypnotize", "Bring\'em Back Alive", "Light My Way", "Getaway Car", "The Last Remaining Light", "Your Time Has Come", "Out Of Exile", "Be Yourself", "Doesn\'t Remind Me", "Drown Me Slowly", "Heaven\'s Dead", "The Worm", "Man Or Animal", "Yesterday To Tomorrow", "Dandelion", "#1 Zero", "The Curse", "Revelations", "One and the Same", "Sound of a Gun", "Until We Fall", "Original Fire", "Broken City", "Somedays", "Shape of Things to Come", "Jewel of the Summertime", "Wide Awake", "Nothing Left to Say But Goodbye", and "Moth".'